In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ML imports für finale Submission
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb
import xgboost as xgb

# Installation der benötigten Pakete (falls noch nicht installiert)
# !pip install lightgbm xgboost

In [3]:
def create_sub(preds, fn=None):
    """Erstellt Submission DataFrame"""
    df = pd.DataFrame({'id': range(len(preds)), 'Prediction': preds})
    return df

In [4]:
def advanced_feature_engineering_v2(df):
    """Erweiterte Feature Engineering für finale Submission"""
    df = df.copy()
    
    # === BASIS ZEIT FEATURES ===
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['weekday'] = df['date'].dt.weekday
    df['day_of_year'] = df['date'].dt.dayofyear
    df['week_of_year'] = df['date'].dt.isocalendar().week
    df['quarter'] = df['date'].dt.quarter
    
    # === ERWEITERTE ZEIT FEATURES ===
    # Wochentag One-Hot
    for i in range(7):
        df[f'is_weekday_{i}'] = (df['weekday'] == i).astype(int)
    
    # Monat One-Hot
    for i in range(1, 13):
        df[f'is_month_{i}'] = (df['month'] == i).astype(int)
    
    # Spezielle Tage
    df['is_weekend'] = (df['weekday'] >= 5).astype(int)
    df['is_monday'] = (df['weekday'] == 0).astype(int)
    df['is_friday'] = (df['weekday'] == 4).astype(int)
    df['is_middle_week'] = ((df['weekday'] >= 1) & (df['weekday'] <= 3)).astype(int)
    
    # === ZYKLISCHE FEATURES ===
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['weekday_sin'] = np.sin(2 * np.pi * df['weekday'] / 7)
    df['weekday_cos'] = np.cos(2 * np.pi * df['weekday'] / 7)
    df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
    df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)
    df['week_sin'] = np.sin(2 * np.pi * df['week_of_year'] / 52)
    df['week_cos'] = np.cos(2 * np.pi * df['week_of_year'] / 52)
    
    # === SAISON FEATURES ===
    df['season'] = df['month'].map({
        12: 0, 1: 0, 2: 0,  # Winter
        3: 1, 4: 1, 5: 1,   # Frühling  
        6: 2, 7: 2, 8: 2,   # Sommer
        9: 3, 10: 3, 11: 3  # Herbst
    })
    
    df['fine_season'] = df['month'].map({
        12: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5,
        6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11
    })
    
    # === WETTER FEATURES ===
    if 'temperature' in df.columns:
        df['temp_very_cold'] = (df['temperature'] < 0).astype(int)
        df['temp_cold'] = ((df['temperature'] >= 0) & (df['temperature'] < 10)).astype(int)
        df['temp_mild'] = ((df['temperature'] >= 10) & (df['temperature'] < 20)).astype(int)
        df['temp_warm'] = ((df['temperature'] >= 20) & (df['temperature'] < 25)).astype(int)
        df['temp_hot'] = (df['temperature'] >= 25).astype(int)
        
        df['temp_squared'] = df['temperature'] ** 2
        df['temp_cubed'] = df['temperature'] ** 3
        df['temp_optimal'] = -((df['temperature'] - 20) ** 2)
        
    if 'precipitation' in df.columns:
        df['no_rain'] = (df['precipitation'] == 0).astype(int)
        df['light_rain'] = ((df['precipitation'] > 0) & (df['precipitation'] <= 2)).astype(int)
        df['moderate_rain'] = ((df['precipitation'] > 2) & (df['precipitation'] <= 10)).astype(int)
        df['heavy_rain'] = (df['precipitation'] > 10).astype(int)
        df['precipitation_log'] = np.log1p(df['precipitation'])
        
    # Wetter-Kombinationen
    if 'temperature' in df.columns and 'precipitation' in df.columns:
        df['perfect_weather'] = ((df['temperature'] > 18) & (df['temperature'] < 25) & 
                                (df['precipitation'] < 0.5)).astype(int)
        df['good_weather'] = ((df['temperature'] > 15) & (df['precipitation'] < 1)).astype(int)
        df['bad_weather'] = ((df['temperature'] < 5) | (df['precipitation'] > 5)).astype(int)
        df['terrible_weather'] = ((df['temperature'] < 0) | (df['precipitation'] > 15)).astype(int)
        df['temp_no_rain'] = df['temperature'] * df['no_rain']
        
    # === FEIERTAG FEATURES ===
    if 'is_public_holiday' in df.columns and 'is_school_holiday' in df.columns:
        df['any_holiday'] = ((df['is_public_holiday'] == 1) | (df['is_school_holiday'] == 1)).astype(int)
        df['both_holidays'] = ((df['is_public_holiday'] == 1) & (df['is_school_holiday'] == 1)).astype(int)
        df['only_school_holiday'] = ((df['is_public_holiday'] == 0) & (df['is_school_holiday'] == 1)).astype(int)
        df['only_public_holiday'] = ((df['is_public_holiday'] == 1) & (df['is_school_holiday'] == 0)).astype(int)
        df['weekend_and_holiday'] = (df['is_weekend'] & df['any_holiday']).astype(int)
        df['workday_and_holiday'] = ((1 - df['is_weekend']) & df['any_holiday']).astype(int)
    
    # === JAHRES-TREND ===
    min_year = df['year'].min()
    df['years_since_start'] = df['year'] - min_year
    df['years_since_start_squared'] = df['years_since_start'] ** 2
    
    # === COVID FEATURE ===
    df['covid_period'] = 0
    
    # 1. Lockdown: März-Mai 2020
    covid_mask1 = ((df['date'] >= '2020-03-22') & (df['date'] <= '2020-05-06'))
    df.loc[covid_mask1, 'covid_period'] = 1
    
    # 2. Lockdown: November 2020 - März 2021
    covid_mask2 = ((df['date'] >= '2020-11-02') & (df['date'] <= '2021-03-07'))
    df.loc[covid_mask2, 'covid_period'] = 2
    
    # Post-COVID
    covid_mask3 = ((df['date'] >= '2021-03-08') & (df['date'] <= '2021-12-31'))
    df.loc[covid_mask3, 'covid_period'] = 3
    
    # COVID One-Hot
    for i in range(4):
        df[f'covid_period_{i}'] = (df['covid_period'] == i).astype(int)
    
    return df

In [5]:
def advanced_outlier_removal(df, target_col='bike_count'):
    """Outlier-Entfernung für finale Submission"""
    if target_col not in df.columns:
        return df
    
    df = df.copy()
    
    # Extreme Outliers
    extreme_threshold = df[target_col].quantile(0.999)
    extreme_outliers = df[target_col] > extreme_threshold
    
    # Negative Werte
    negative_outliers = df[target_col] < 0
    
    # Saisonale Outliers
    seasonal_outliers = pd.Series(False, index=df.index)
    for month in range(1, 13):
        month_data = df[df['date'].dt.month == month][target_col]
        if len(month_data) > 10:
            Q1 = month_data.quantile(0.25)
            Q3 = month_data.quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 3.0 * IQR
            upper_bound = Q3 + 3.0 * IQR
            
            month_outliers = (df['date'].dt.month == month) & \
                           ((df[target_col] < lower_bound) | (df[target_col] > upper_bound))
            seasonal_outliers = seasonal_outliers | month_outliers
    
    all_outliers = extreme_outliers | negative_outliers | seasonal_outliers
    
    before_count = len(df)
    df_clean = df[~all_outliers]
    after_count = len(df_clean)
    
    print(f"Outlier entfernt: {before_count - after_count} von {before_count} ({100*(before_count-after_count)/before_count:.1f}%)")
    
    return df_clean


In [6]:
def train_final_models(X_train, y_train, X_test):
    """Trainiert die 4 Modelle für das finale WeightedEnsemble"""
    
    models = {}
    predictions = {}
    cv_scores = {}
    
    tscv = TimeSeriesSplit(n_splits=5)
    
    print("=== TRAINING FINAL MODELS FOR WEIGHTED ENSEMBLE ===")
    
    # 1. LightGBM
    print("1. LightGBM...")
    lgb_params = {
        'objective': 'regression',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.05,
        'feature_fraction': 0.9,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1,
        'random_state': 42
    }
    
    lgb_model = lgb.LGBMRegressor(**lgb_params, n_estimators=1000)
    lgb_model.fit(X_train, y_train, eval_set=[(X_train, y_train)], 
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    
    models['LightGBM'] = lgb_model
    predictions['LightGBM'] = lgb_model.predict(X_test)
    
    cv_scores_lgb = -cross_val_score(lgb_model, X_train, y_train, cv=tscv, scoring='neg_mean_absolute_error')
    cv_scores['LightGBM'] = cv_scores_lgb.mean()
    print(f"   CV MAE: {cv_scores_lgb.mean():.2f}")
    
    # 2. XGBoost
    print("2. XGBoost...")
    xgb_params = {
        'objective': 'reg:absoluteerror',
        'max_depth': 6,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': 42,
        'n_estimators': 1000
    }
    
    xgb_model = xgb.XGBRegressor(**xgb_params)
    xgb_model.fit(X_train, y_train, eval_set=[(X_train, y_train)], 
                  early_stopping_rounds=50, verbose=False)
    
    models['XGBoost'] = xgb_model
    predictions['XGBoost'] = xgb_model.predict(X_test)
    
    cv_scores_xgb = -cross_val_score(xgb_model, X_train, y_train, cv=tscv, scoring='neg_mean_absolute_error')
    cv_scores['XGBoost'] = cv_scores_xgb.mean()
    print(f"   CV MAE: {cv_scores_xgb.mean():.2f}")
    
    # 3. Extra Trees
    print("3. Extra Trees...")
    et_model = ExtraTreesRegressor(
        n_estimators=500,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
    et_model.fit(X_train, y_train)
    
    models['ExtraTrees'] = et_model
    predictions['ExtraTrees'] = et_model.predict(X_test)
    
    cv_scores_et = -cross_val_score(et_model, X_train, y_train, cv=tscv, scoring='neg_mean_absolute_error')
    cv_scores['ExtraTrees'] = cv_scores_et.mean()
    print(f"   CV MAE: {cv_scores_et.mean():.2f}")
    
    # 4. Gradient Boosting
    print("4. Gradient Boosting...")
    gb_model = GradientBoostingRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=4,
        subsample=0.8,
        random_state=42
    )
    gb_model.fit(X_train, y_train)
    
    models['GradientBoosting'] = gb_model
    predictions['GradientBoosting'] = gb_model.predict(X_test)
    
    cv_scores_gb = -cross_val_score(gb_model, X_train, y_train, cv=tscv, scoring='neg_mean_absolute_error')
    cv_scores['GradientBoosting'] = cv_scores_gb.mean()
    print(f"   CV MAE: {cv_scores_gb.mean():.2f}")
    
    return models, predictions, cv_scores

In [7]:
def create_weighted_ensemble(predictions, cv_scores):
    """Erstellt das finale WeightedEnsemble"""
    
    # Gewichte basierend auf inverser CV-MAE
    weights = {}
    total_inverse_mae = 0
    
    for model_name, mae in cv_scores.items():
        if model_name in predictions:
            inverse_mae = 1.0 / mae
            weights[model_name] = inverse_mae
            total_inverse_mae += inverse_mae
    
    # Normalisiere Gewichte
    for model_name in weights:
        weights[model_name] /= total_inverse_mae
    
    print(f"\nWeighted Ensemble Gewichte:")
    for model_name, weight in weights.items():
        print(f"  {model_name}: {weight:.3f}")
    
    # Gewichteter Durchschnitt
    ensemble_pred = np.zeros(len(next(iter(predictions.values()))))
    for model_name, pred in predictions.items():
        if model_name in weights:
            ensemble_pred += weights[model_name] * pred
    
    return ensemble_pred

In [8]:

# ========================================
# HAUPTCODE 
# ========================================

print("=== FINALE WEIGHTED ENSEMBLE SUBMISSION ===\n")

# 1. Daten laden
print("1. Lade Daten...")
df_train = pd.read_csv("/kaggle/input/umr-ml-2025-hackathon-1/data_train.csv")
df_test = pd.read_csv("/kaggle/input/umr-ml-2025-hackathon-1/data_test.csv")

df_train['date'] = pd.to_datetime(df_train['date'])
df_test['date'] = pd.to_datetime(df_test['date'])

print(f"Training Daten: {len(df_train)} Zeilen")
print(f"Test Daten: {len(df_test)} Zeilen")

# 2. Outlier-Entfernung
print("\n2. Outlier-Entfernung...")
df_train_clean = advanced_outlier_removal(df_train)

# 3. Feature Engineering
print("\n3. Feature Engineering...")
df_train_features = advanced_feature_engineering_v2(df_train_clean)
df_test_features = advanced_feature_engineering_v2(df_test)

# 4. Features vorbereiten
feature_cols = [col for col in df_train_features.columns 
               if col not in ['date', 'bike_count']]

X_train = df_train_features[feature_cols]
y_train = df_train_features['bike_count']
X_test = df_test_features[feature_cols]

print(f"Features: {len(feature_cols)}")

# 5. Modelle trainieren
print("\n4. Trainiere Modelle...")
models, predictions, cv_scores = train_final_models(X_train, y_train, X_test)

# 6. WeightedEnsemble erstellen
print("\n5. Erstelle WeightedEnsemble...")
weighted_ensemble_pred = create_weighted_ensemble(predictions, cv_scores)

# 7. Finale Submission erstellen
print("\n6. Erstelle Submission...")

# Keine negativen Vorhersagen
final_predictions = np.maximum(weighted_ensemble_pred, 0)

# Erstelle finale Submission
df_final_submission = create_sub(final_predictions)
df_final_submission.to_csv('submission_advanced_weightedensemble.csv', index=False)

print("✓ submission_advanced_weightedensemble.csv erstellt")

# 8. Finale Statistiken
print("\n=== FINALE ERGEBNISSE ===")
print("Individual Model CV MAE:")
for model_name, mae in sorted(cv_scores.items(), key=lambda x: x[1]):
    print(f"  {model_name:15s}: {mae:.2f}")

estimated_ensemble_mae = np.mean(list(cv_scores.values()))
print(f"\nEstimated WeightedEnsemble MAE: ~{estimated_ensemble_mae:.2f}")
print(f"Vorhersage-Bereich: {final_predictions.min():.0f} - {final_predictions.max():.0f}")
print(f"Durchschnittliche Vorhersage: {final_predictions.mean():.0f}")


=== FINALE WEIGHTED ENSEMBLE SUBMISSION ===

1. Lade Daten...
Training Daten: 3901 Zeilen
Test Daten: 390 Zeilen

2. Outlier-Entfernung...
Outlier entfernt: 6 von 3901 (0.2%)

3. Feature Engineering...
Features: 78

4. Trainiere Modelle...
=== TRAINING FINAL MODELS FOR WEIGHTED ENSEMBLE ===
1. LightGBM...
   CV MAE: 610.77
2. XGBoost...
   CV MAE: 614.12
3. Extra Trees...
   CV MAE: 639.86
4. Gradient Boosting...
   CV MAE: 615.96

5. Erstelle WeightedEnsemble...

Weighted Ensemble Gewichte:
  LightGBM: 0.254
  XGBoost: 0.252
  ExtraTrees: 0.242
  GradientBoosting: 0.252

6. Erstelle Submission...
✓ submission_advanced_weightedensemble.csv erstellt

=== FINALE ERGEBNISSE ===
Individual Model CV MAE:
  LightGBM       : 610.77
  XGBoost        : 614.12
  GradientBoosting: 615.96
  ExtraTrees     : 639.86

Estimated WeightedEnsemble MAE: ~620.17
Vorhersage-Bereich: 398 - 7229
Durchschnittliche Vorhersage: 4349
